<a href="https://colab.research.google.com/github/qurban-12/ML_internship_Qurban/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qurban-12/ML_internship_Qurban/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### My baseline rule

I will prioritize content using two observed signals:

* **Stale content:** `days_since_last_update >= 91`
* **CTR opportunity:** `avg_position` is between 1 and 10, while `ctr < 0.10%`
* **Both signals:** content meets both conditions and receives the highest priority.
* **No clear signal:** content meets neither condition and receives the lowest priority.

### Reason codes

* `STALE_CONTENT` — content has not been updated for at least 91 days.
* `LOW_CTR_GOOD_POSITION` — content has a good average search position but low CTR.
* `STALE_AND_LOW_CTR` — both signals are present.
* `NO_CLEAR_SIGNAL` — neither signal is present.

The score is a transparent, rule-based baseline. It does not use `trend_direction`, `trend_pct`, the decline label, future-window information, client names, or product flags as model inputs.


In [33]:
# Baseline rule thresholds

STALE_DAYS = 91
GOOD_POSITION_MAX = 10
LOW_CTR_MAX = 0.10

print("Baseline rule thresholds:")
print(f"- Stale content: >= {STALE_DAYS} days")
print(f"- Good position: 1-{GOOD_POSITION_MAX}")
print(f"- Low CTR: < {LOW_CTR_MAX}%")

Baseline rule thresholds:
- Stale content: >= 91 days
- Good position: 1-10
- Low CTR: < 0.1%


In [34]:
from pathlib import Path
import pandas as pd

# Repository and dataset paths
REPO_PATH = Path("/content/ML_internship_Qurban")
DATA_PATH = REPO_PATH / "data/raw/content_refresh_anonymized.csv"

# Check that the repository and dataset exist
if not REPO_PATH.exists():
    raise FileNotFoundError(f"Repository not found: {REPO_PATH}")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Dataset loaded successfully.")

Dataset shape: (30000, 44)
Dataset loaded successfully.


In [35]:
# Columns required for our baseline signals
required_columns = [
    "days_since_last_update",
    "ctr",
    "avg_position"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Required columns found:")
for col in required_columns:
    print("-", col)

Required columns found:
- days_since_last_update
- ctr
- avg_position


In [36]:
print("--- Staleness signal ---")
print(df["days_since_last_update"].describe())

print("\n--- CTR signal ---")
print(df["ctr"].describe())

print("\n--- Average position signal ---")
print(df["avg_position"].describe())

--- Staleness signal ---
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

--- CTR signal ---
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

--- Average position signal ---
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64


In [37]:
# Create transparent freshness buckets
def freshness_bucket(days):
    if days <= 30:
        return "0-30"
    elif days <= 90:
        return "31-90"
    elif days <= 180:
        return "91-180"
    else:
        return "181+"

df["freshness_bucket"] = df["days_since_last_update"].apply(
    freshness_bucket
)

print("Freshness buckets created.")
print(df["freshness_bucket"].value_counts().sort_index())

Freshness buckets created.
freshness_bucket
0-30      20480
181+        174
31-90       175
91-180     9171
Name: count, dtype: int64


In [38]:
# Signal 1 audit: staleness distribution

staleness_audit = (
    df.groupby("freshness_bucket")
      .agg(
          n=("days_since_last_update", "size"),
          median_days=("days_since_last_update", "median"),
          max_days=("days_since_last_update", "max")
      )
      .reset_index()
)

print("Signal 1: Staleness / Freshness")
print(staleness_audit.to_string(index=False))

print("\nOverall staleness percentiles:")
print(
    df["days_since_last_update"]
      .quantile([0.25, 0.50, 0.75, 0.90, 0.95])
      .to_string()
)

Signal 1: Staleness / Freshness
freshness_bucket     n  median_days  max_days
            0-30 20480         20.0        28
            181+   174        211.0       373
           31-90   175         41.0        89
          91-180  9171        104.0       151

Overall staleness percentiles:
0.25     20.0
0.50     20.0
0.75    104.0
0.90    104.0
0.95    104.0


### Signal 1 verdict — Staleness

**Verdict: CONFIRMED**

The observed data shows a clear freshness structure: most content was updated within 30 days, while a smaller group has gone beyond 90 days since the last update. I will therefore treat content with 91 or more days since its last update as the stale-content signal.

This is a directional baseline rule, not a claim that every old page needs a refresh.

In [39]:
# Signal 2 audit: CTR by average-position bucket

def position_bucket(position):
    if position <= 0:
        return "no_position"
    elif position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    elif position <= 50:
        return "21-50"
    else:
        return "51+"

df["position_bucket"] = df["avg_position"].apply(position_bucket)

ctr_position_audit = (
    df.groupby("position_bucket")
      .agg(
          n=("ctr", "size"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

ctr_position_audit["median_ctr"] = (
    ctr_position_audit["median_ctr"].round(3)
)

ctr_position_audit["mean_ctr"] = (
    ctr_position_audit["mean_ctr"].round(3)
)

print("Signal 2: CTR vs. Average Position")
print(ctr_position_audit.to_string(index=False))

Signal 2: CTR vs. Average Position
position_bucket     n  median_ctr  mean_ctr
            1-3  1141        0.00     2.714
          11-20  7273        0.10     0.323
          21-50  7225        0.03     0.222
           4-10 11842        0.16     0.651
            51+  1314        0.00     0.151
    no_position  1205        0.00     0.297


### Signal 2 verdict — CTR vs. Position

**Verdict: CONFIRMED**

The observed CTR varies across position buckets. Pages in positions 1–10 can still have very low CTR, while pages at weaker positions generally have lower median CTR. I will use a conservative baseline condition of average position <= 10 with CTR < 0.10.

This is treated as a CTR opportunity signal, not proof that changing the page will increase CTR.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [41]:
# Build the baseline action score

df["is_stale"] = (
    df["days_since_last_update"] >= STALE_DAYS
)

df["is_low_ctr_good_position"] = (
    (df["avg_position"] >= 1)
    & (df["avg_position"] <= GOOD_POSITION_MAX)
    & (df["ctr"] < LOW_CTR_MAX)
)

# Score: both signals > stale only > CTR opportunity only > no signal
df["baseline_score"] = (
    df["is_stale"].astype(int) * 2
    + df["is_low_ctr_good_position"].astype(int)
)

# Reason code
def get_reason_code(row):
    if row["is_stale"] and row["is_low_ctr_good_position"]:
        return "STALE_AND_LOW_CTR"
    elif row["is_stale"]:
        return "STALE_CONTENT"
    elif row["is_low_ctr_good_position"]:
        return "LOW_CTR_GOOD_POSITION"
    else:
        return "NO_CLEAR_SIGNAL"

df["reason_code"] = df.apply(get_reason_code, axis=1)

# Action label
def get_action(score):
    if score == 3:
        return "refresh_and_ctr_fix"
    elif score == 2:
        return "refresh"
    elif score == 1:
        return "ctr_fix"
    else:
        return "monitor"

df["action"] = df["baseline_score"].apply(get_action)

print("Baseline scoring completed.")
print("\nScore distribution:")
print(df["baseline_score"].value_counts().sort_index())

print("\nReason code distribution:")
print(df["reason_code"].value_counts())

Baseline scoring completed.

Score distribution:
baseline_score
0    16532
1     4123
2     7933
3     1412
Name: count, dtype: int64

Reason code distribution:
reason_code
NO_CLEAR_SIGNAL          16532
STALE_CONTENT             7933
LOW_CTR_GOOD_POSITION     4123
STALE_AND_LOW_CTR         1412
Name: count, dtype: int64


In [42]:
# Rank the baseline action queue

output_columns = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

missing_output_columns = [
    col for col in output_columns
    if col not in df.columns
]

if missing_output_columns:
    raise ValueError(
        f"Missing output columns: {missing_output_columns}"
    )

ranked_queue = (
    df[output_columns]
    .sort_values(
        by=[
            "baseline_score",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "rank",
    range(1, len(ranked_queue) + 1)
)

output_path = REPO_PATH / "work/outputs/baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print(f"Ranked queue written to: {output_path}")
print(f"Rows written: {len(ranked_queue)}")

print("\nTop 10 rows:")
print(ranked_queue.head(10).to_string(index=False))

Ranked queue written to: /content/ML_internship_Qurban/work/outputs/baseline_action_score.csv
Rows written: 30000

Top 10 rows:
 rank           content_id  baseline_score       reason_code              action  days_since_last_update  ctr  avg_position
    1 content_55a5b1c46474               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     373  0.0           7.5
    2 content_1b4ec72dafd4               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     372  0.0           7.0
    3 content_06e19c6486b0               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     334  0.0           5.0
    4 content_e2b702f4f92b               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     334  0.0           9.3
    5 content_02b0d6e30129               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     313  0.0           6.9
    6 content_f488400fca67               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     305  0.0           5.7
    

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [44]:
# Show the top 20 ranked items for manual review

top20 = ranked_queue.head(20).copy()

print("Top-20 baseline action queue:")
print(
    top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ].to_string(index=False)
)

Top-20 baseline action queue:
 rank           content_id  baseline_score       reason_code              action  days_since_last_update  ctr  avg_position
    1 content_55a5b1c46474               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     373  0.0           7.5
    2 content_1b4ec72dafd4               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     372  0.0           7.0
    3 content_06e19c6486b0               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     334  0.0           5.0
    4 content_e2b702f4f92b               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     334  0.0           9.3
    5 content_02b0d6e30129               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     313  0.0           6.9
    6 content_f488400fca67               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     305  0.0           5.7
    7 content_7736e6144f3b               3 STALE_AND_LOW_CTR refresh_and_ctr_fix                     3

### Top-20 manual review

The top 20 items all receive the highest baseline score because they meet both rule conditions: at least 91 days since the last update and CTR below 0.10% while average position is between 1 and 10.

| Rank | Action            | Why it is here                          | What would make it wrong                                                        |
| ---: | ----------------- | --------------------------------------- | ------------------------------------------------------------------------------- |
|    1 | Refresh + CTR fix | 373 days stale, CTR 0.00%, position 7.5 | The page may no longer be important or the CTR measurement may be unreliable.   |
|    2 | Refresh + CTR fix | 372 days stale, CTR 0.00%, position 7.0 | The page may have low current value or insufficient search activity.            |
|    3 | Refresh + CTR fix | 334 days stale, CTR 0.00%, position 5.0 | The page may not have enough meaningful search demand despite its position.     |
|    4 | Refresh + CTR fix | 334 days stale, CTR 0.00%, position 9.3 | The low CTR may reflect limited impressions or another measurement limitation.  |
|    5 | Refresh + CTR fix | 313 days stale, CTR 0.00%, position 6.9 | The page may not be a current business priority.                                |
|    6 | Refresh + CTR fix | 305 days stale, CTR 0.00%, position 5.7 | The page may have limited measurable opportunity.                               |
|    7 | Refresh + CTR fix | 304 days stale, CTR 0.00%, position 5.2 | The CTR signal may not represent a meaningful optimization opportunity.         |
|    8 | Refresh + CTR fix | 304 days stale, CTR 0.00%, position 5.3 | The page may have insufficient search activity for a useful CTR decision.       |
|    9 | Refresh + CTR fix | 304 days stale, CTR 0.00%, position 8.0 | The page may have limited current relevance or opportunity.                     |
|   10 | Refresh + CTR fix | 304 days stale, CTR 0.00%, position 8.9 | The low CTR may not be actionable without additional query-level evidence.      |
|   11 | Refresh + CTR fix | 301 days stale, CTR 0.00%, position 6.4 | The page may not have enough measurable search demand.                          |
|   12 | Refresh + CTR fix | 231 days stale, CTR 0.00%, position 1.3 | Position alone does not prove that a CTR improvement is possible.               |
|   13 | Refresh + CTR fix | 231 days stale, CTR 0.00%, position 4.5 | The page may have limited impressions or search demand.                         |
|   14 | Refresh + CTR fix | 231 days stale, CTR 0.00%, position 5.0 | The observed CTR may not be stable enough to support an action.                 |
|   15 | Refresh + CTR fix | 211 days stale, CTR 0.00%, position 2.0 | A strong position does not guarantee a meaningful CTR opportunity.              |
|   16 | Refresh + CTR fix | 211 days stale, CTR 0.00%, position 2.3 | The page may have insufficient impressions or relevance.                        |
|   17 | Refresh + CTR fix | 211 days stale, CTR 0.00%, position 3.0 | The low CTR could reflect query or SERP context not captured by this baseline.  |
|   18 | Refresh + CTR fix | 211 days stale, CTR 0.00%, position 3.0 | The signal may not translate into an actionable page-level change.              |
|   19 | Refresh + CTR fix | 211 days stale, CTR 0.00%, position 3.0 | Additional search-demand evidence could show limited opportunity.               |
|   20 | Refresh + CTR fix | 211 days stale, CTR 0.00%, position 3.0 | The page may not be worth prioritizing despite meeting the baseline conditions. |

### Review summary

The main weakness of these picks is that the rule treats **CTR = 0.00%** as a strong CTR opportunity without checking the amount of search activity behind the measurement. A page with very few impressions can have a zero CTR without providing enough evidence for a CTR-focused action.

Therefore, these are **directional decision-support picks**, not confirmed optimization opportunities.


In [45]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak-pick and leakage checks

print("=== Weak-pick check ===")

weak_pick_conditions = (
    (ranked_queue["reason_code"] == "STALE_AND_LOW_CTR")
    & (ranked_queue["ctr"] == 0)
)

weak_pick_count = weak_pick_conditions.sum()

print(
    f"Top-20 items with CTR = 0.00%: "
    f"{weak_pick_conditions.head(20).sum()} / 20"
)

print(
    "\nPotential weakness: the baseline does not currently "
    "require a minimum number of impressions before treating "
    "low CTR as an actionable opportunity."
)

print("\n=== Leakage check ===")

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_columns = set(ranked_queue.columns)

leaked_columns = [
    col for col in forbidden_features
    if col in used_columns
]

if leaked_columns:
    raise ValueError(
        f"Potential leakage detected in output: {leaked_columns}"
    )

print("No trend/label-derived columns are present in the ranked queue.")

print("\nRule input columns used:")
print("- days_since_last_update")
print("- ctr")
print("- avg_position")

print("\nLeakage check passed.")

=== Weak-pick check ===
Top-20 items with CTR = 0.00%: 20 / 20

Potential weakness: the baseline does not currently require a minimum number of impressions before treating low CTR as an actionable opportunity.

=== Leakage check ===
No trend/label-derived columns are present in the ranked queue.

Rule input columns used:
- days_since_last_update
- ctr
- avg_position

Leakage check passed.


### Weak picks

The main weak-pick concern is the CTR signal. All 20 top-ranked items have CTR of 0.00%, but the baseline does not currently require a minimum number of impressions before treating zero CTR as an actionable opportunity.

This means some pages could be ranked highly because of a zero CTR measurement even when the available search activity is too small to support a strong decision.

### Leakage check

The ranked queue uses only:

* `days_since_last_update`
* `ctr`
* `avg_position`

The queue does not use `trend_direction`, `trend_pct`, or `is_declining_label`.

No future-window or label-derived features were used in the baseline rule.

**Leakage check: PASSED.**

### Overall baseline assessment

This baseline is transparent and directional. It provides a simple ranked queue for review using observable staleness and CTR-versus-position signals. The Top-20 review also shows an important limitation: CTR should ideally be considered together with the amount of measurable search activity before making an action decision.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.